# 4. On-Behalf-Of (OBO) — preserving user identity across tiers

**Scenario**: a user Alice signs into a frontend, which calls API-A (middle tier). API-A needs to call API-B, but API-B must still know that **Alice** is the one asking — so it can apply her row-level permissions.

Options:

| Option | Who does API-B think is calling? | Row-level security? |
|--------|----------------------------------|---------------------|
| Client credentials | API-A (the app) | No — API-B sees an app, not Alice |
| Forward Alice's token | Alice, but `aud` = API-A, fails validation at API-B | Broken |
| **On-Behalf-Of** | Alice — a fresh token `aud=api://api-b`, `upn=alice@...` | ✅ |

OBO is a special grant type where API-A exchanges Alice's token for a new token targeting API-B, while preserving her identity claims.

## The flow

```
Alice → /proxy/files  (Bearer: alice-token for api://api-a)
         │
         └── API-A → POST /token  grant=jwt-bearer
                      assertion=alice-token
                      scope=api://api-b/Files.Read
                      client_id=api-a  client_secret=...
                      requested_token_use=on_behalf_of
                    ← new token  aud=api://api-b  upn=alice  scp=Files.Read
         │
         └── GET http://api-b/files   (Bearer: new token)
                     → returns only Alice's files
```

Key points:

- API-A proves **itself** to Entra (client_id + secret) — Entra won't let any app swap tokens.
- The `assertion` is the user's original token. Entra validates it was issued for API-A.
- Entra returns a **new** token with a different `aud` but the same user identity.
- This requires that API-A has been granted the `Files.Read` delegated permission on api-b, *and* the user (or an admin) has consented.

In [ ]:
import httpx, json, base64

TOKEN_URL = 'http://localhost:9000/contoso/oauth2/v2.0/token'
API_A     = 'http://localhost:8001'
API_B     = 'http://localhost:8002'

def decode(t):
    p = t.split('.')[1]
    return json.loads(base64.urlsafe_b64decode(p + '=' * (-len(p) % 4)))

# --- Step 1: Alice signs in (ROPC - demo only; production uses auth code flow) ---
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'password',
    'client_id': 'api-a-client-id',
    'client_secret': 'api-a-secret-value',
    'username': 'alice@contoso.com',
    'password': 'alice-password',
    'scope': 'api://api-a/access_as_user',
})
alice_token = r.json()['access_token']
print('Alice\'s token (aud=api-a):')
print(json.dumps(decode(alice_token), indent=2))

> ⚠️ In production you'd get `alice_token` through the **authorization code** flow — a browser redirect. ROPC is used here for brevity and **is not supported in real Entra for multi-factor accounts**.

In [ ]:
# --- Step 2: Alice calls API-A ---
r = httpx.get(f'{API_A}/proxy/files', headers={'Authorization': f'Bearer {alice_token}'})
print(json.dumps(r.json(), indent=2))

Look at the response:
- `api_a_saw_user` = `alice@contoso.com` (API-A validated her token)
- Inside `api_b_response`:
  - `mode: delegated` (API-B saw a user token, not app-only)
  - `user: alice@contoso.com` (preserved via OBO)
  - Only Alice's files — bob's are filtered out

## What API-A actually did

Inspect [`api-a/server.py`](../api-a/server.py) — specifically `_exchange_obo`. Or reproduce it by hand:

In [ ]:
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'urn:ietf:params:oauth:grant-type:jwt-bearer',
    'client_id': 'api-a-client-id',
    'client_secret': 'api-a-secret-value',
    'assertion': alice_token,
    'scope': 'api://api-b/Files.Read',
    'requested_token_use': 'on_behalf_of',
})
downstream = r.json()['access_token']
print('Downstream token (aud=api-b, user=alice):')
print(json.dumps(decode(downstream), indent=2))

Compare with Alice's original token — same `oid`, same `upn`, **different `aud`** and **different `scp`**.

## Compare to client credentials

Same user hits API-A, but API-A uses client credentials instead:

In [ ]:
r = httpx.get(f'{API_A}/proxy/files/admin', headers={'Authorization': f'Bearer {alice_token}'})
print(json.dumps(r.json(), indent=2))

Now API-B returns **all three** files. `mode: app-only` because the downstream token has `roles`, no `upn`. API-B can't enforce Alice's permissions because it doesn't know who she is.

**Use OBO when you need row-level security or per-user auditing.** Use client credentials for system-level operations (batch jobs, admin tools).

## Common OBO pitfalls

| Error | Cause |
|-------|-------|
| `AADSTS50013: Assertion failed signature validation` | Incoming token was for a *different* audience than your middle tier. It must be `api://api-a`. |
| `AADSTS65001: The user has not consented` | User (or admin) hasn't consented to the downstream scope. Add it to API-A's API permissions. |
| `AADSTS50105: Signed in user not assigned to application role` | Assignment required on the enterprise app — add the user or the group. |
| OBO returns a token but API-B rejects it | Check `aud` in the downstream token matches what API-B validates. |

## Configuring OBO in real Entra

1. On **API-B** app registration → *Expose an API* → add scope `Files.Read`.
2. On **API-A** app registration → *API permissions* → add delegated `Files.Read` on API-B. Grant admin consent.
3. On **API-A** → *Expose an API* → add scope `access_as_user`. The frontend requests this when the user signs in.
4. In code — use MSAL: `ConfidentialClientApplication.acquire_token_on_behalf_of(user_assertion=alice_token, scopes=['api://api-b/Files.Read'])`.

## Summary

- OBO preserves *user* identity across service hops.
- The middle tier must prove itself with its own creds + present the user's token as the `assertion`.
- Downstream token has `upn`/`scp`, same `oid` as original.
- Pick OBO when per-user authorization matters; pick client credentials for system tasks.